### Домашнее задание 5 - 15 баллов

1. **Выбор датасета - 1 балл**
   - Выберите и загрузите датасет для задачи генерации текста 
   - Подходящие варианты: 
     * Текстовые корпусы на платформе Hugging Face
     * Ваши собственные данные
     
2. **Выбор предобученной модели - 1 балл**
   - Выберите подходящую (по размеру - от 0.5B-1.5B параметров) предобученную языковую модель на Hugging Face Model Hub

3. **Предварительная оценка качества - 1 балл**
   - Соберите небольшую "корзинку" тестовых примеров, прогоните их через модель для оценки качества генерации перед дообучением
   - Дополнительно прогоните модель через выбранную вами одну или несколько релевантных задач из lm-evaluation-harness

4. **QLora-дообучение - 2 балла**
   - Настройте параметры дообучения через QLora, опишите свой выбор значений и настраиваемых параметров в комментариях
   - Обучите модель на вашем датасете

5. **Оценка качества обучения - 1 балл**
   - Проверьте качество генерации на бенчмарке и "корзинке" после дообучения

6. **Профайлинг обучения - 5 баллов**
   - Оберните несколько шагов обучения в torch.profiler.profile
   - Проанализируйте логи профайлера или трейсы, сделайте выводы о процессе обучения, узких местах



**Общее**

- Принимаемые решения обоснованы (почему выбрана определенная архитектура/гиперпараметр/оптимизатор/преобразование и т.п.), по ходу работы присутствуют комментарии и выводы - **3 балла**
- Обеспечена воспроизводимость решения: зафиксированы random_state, ноутбук воспроизводится от начала до конца без ошибок - **1 балл**

**Базовая модель:** Qwen/Qwen2.5-0.5B-Instruct  
**Датасет:** IlyaGusev/ru_turbo_alpaca

In [2]:
! pip -q install -U transformers datasets peft accelerate bitsandbytes trl sentencepiece evaluate lm-eval langdetect

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ml-core-dal-client-autoconf 0.0.6 requires opentelemetry-api==1.39.1, but you have opentelemetry-api 1.40.0 which is incompatible.

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [8]:
import os
import gc
import json
import random
import warnings

import numpy as np
import torch

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATASET_NAME = "IlyaGusev/ru_turbo_alpaca"

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

torch: 2.11.0+cu130
cuda available: True


In [5]:
#raw файл датасета
from huggingface_hub import hf_hub_download

data_path = hf_hub_download(
    repo_id=DATASET_NAME,
    repo_type="dataset",
    filename="ru_turbo_alpaca.jsonl.zst",
)

print(data_path)

ru_turbo_alpaca.jsonl.zst:   0%|          | 0.00/14.6M [00:00<?, ?B/s]

/home/mlcore/.cache/huggingface/hub/datasets--IlyaGusev--ru_turbo_alpaca/snapshots/460b1f3312aa21ef774e916e532a9576f7938a0d/ru_turbo_alpaca.jsonl.zst


In [12]:
#пример из датасета
from datasets import load_dataset

raw_ds = load_dataset(
    "json",
    data_files=data_path,
)

train_ds = raw_ds["train"]

print(raw_ds)

for k, v in train_ds[0].items():
    text = str(v)
    print(f"{k}: {text[:500]}", "...")

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'label', 'all_labels', 'overlap', 'agreement', 'alternative_output'],
        num_rows: 29822
    })
})
instruction: Опишите, как сделать горшок из глины. ...
input:  ...
output: Для изготовления горшка из глины сначала необходимо взять глину и размешать ее водой до состояния пластилина. Затем нужно сформировать глиняный комок нужной формы и тщательно откачать влагу из глины, чтобы она стала твердой. Горшок нужно обжечь в духовке при высокой температуре, чтобы он стал прочным и непроницаемым. ...
label: bad_output ...
all_labels: ['bad_output', 'bad_output'] ...
overlap: 2 ...
agreement: 1.0 ...
alternative_output: Для изготовления горшка из глины нужно выполнить следующие шаги:

1. Подготовить глину: глину необходимо хорошо промыть и измельчить. Если глина слишком сухая, ее нужно замочить.

2. Сформировать горшок: возьмите кусок глины и разминайте его в руках, сделав шарик. Потом начинайте вращать 

In [13]:
#выбираем лучший ответ
def pick_response(example):
    use_alt = example["label"] == "bad_output" and example["alternative_output"] not in [None, ""]
    answer = example["alternative_output"] if use_alt else example["output"]
    return {
        "instruction": (example["instruction"] or "").strip(),
        "input": (example["input"] or "").strip(),
        "answer": (answer or "").strip(),
    }

ds = train_ds.map(pick_response, remove_columns=train_ds.column_names)

ds = ds.filter(lambda x: len(x["instruction"]) > 0 and len(x["answer"]) > 0)

print(ds)
print(ds[0])

Map:   0%|          | 0/29822 [00:00<?, ? examples/s]

Filter:   0%|          | 0/29822 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'answer'],
    num_rows: 29822
})
{'instruction': 'Опишите, как сделать горшок из глины.', 'input': '', 'answer': 'Для изготовления горшка из глины нужно выполнить следующие шаги:\n\n1. Подготовить глину: глину необходимо хорошо промыть и измельчить. Если глина слишком сухая, ее нужно замочить.\n\n2. Сформировать горшок: возьмите кусок глины и разминайте его в руках, сделав шарик. Потом начинайте вращать шарик между ладонями, чтоб сформировался цилиндр. Затем используйте пальцы, чтобы создать стенки горшка высотой и диаметром по желанию.\n\n3. Сделать отверстие для дренажа: вы сможете сделать отверстие для дренажа для того, чтобы вода смогла выходить из горшка, когда он находится на подставке. Для этого, воспользуйтесь ножом или другим инструментом для делания отверстия на дне горшка.\n\n4. Высушить горшок: дайте горшку высохнуть на протяжении 24 часов при комнатной температуре.\n\n5. Обжиг горшка: после того как горшок высохнет, следует

In [ ]:
SYSTEM_PROMPT = (
    "Ты полезный русскоязычный ассистент. "
    "Отвечай ясно, по делу и естественным языком. "
    "Следуй просьбе пользователя, не уходи в сторону. "
    "Если уместно, давай ответ в виде короткого списка шагов."
)

In [53]:
#chat формат для квена
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

def to_text(example):
    user_text = example["instruction"]
    if example["input"]:
        user_text += "\n\nКонтекст:\n" + example["input"]

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
        {"role": "assistant", "content": example["answer"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {
        "messages": messages,
        "text": text,
    }

ds = ds.remove_columns(["messages", "text"]) if "messages" in ds.column_names else ds
ds = ds.map(to_text)

print(ds)
print()
print(ds[0]["text"][:500], "...")

Map:   0%|          | 0/29822 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'answer', 'messages', 'text'],
    num_rows: 29822
})

<|im_start|>system
Ты полезный русскоязычный ассистент. Отвечай ясно, по делу и естественным языком. Следуй просьбе пользователя, не уходи в сторону. Если уместно, давай ответ в виде короткого списка шагов.<|im_end|>
<|im_start|>user
Опишите, как сделать горшок из глины.<|im_end|>
<|im_start|>assistant
Для изготовления горшка из глины нужно выполнить следующие шаги:

1. Подготовить глину: глину необходимо хорошо промыть и измельчить. Если глина слишком сухая, ее нужно замочить.

2. Сформировать  ...


In [54]:
#split и тестовые промпты (18 шт)
from datasets import Dataset
import pandas as pd

split = ds.train_test_split(test_size=0.02, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

test_prompts = [
    "Объясни простыми словами, как работает Wi-Fi дома.",
    "Напиши короткое и вежливое сообщение коллеге с просьбой перенести встречу на завтра.",
    "Подскажи, как распределить домашние дела, чтобы убрать квартиру за один вечер.",
    "Объясни, как правильно сварить макароны, чтобы они не слиплись.",
    "Составь план подготовки к поездке в горы на выходные.",
    "Подскажи, как выбрать недорогой подарок другу на день рождения.",
    "Объясни, как хранить овощи и фрукты, чтобы они дольше не портились.",
    "Напиши короткую инструкцию, что сделать перед выходом из дома в дождливую погоду.",
    "Подскажи, как ухаживать за комнатным растением начинающему.",
    "Объясни, как быстро остудить горячий чай.",
    "Подскажи, что взять с собой в спортзал.",
    "Объясни, как безопасно разморозить мясо или курицу.",
    "Напиши короткое поздравление с днем рождения в дружелюбном тоне.",
    "Составь пошаговый план, как приготовить простой завтрак за 15 минут."
]

test_basket = Dataset.from_dict({"prompt": test_prompts})

print(train_dataset)
print(eval_dataset)
pd.DataFrame({"prompt": test_prompts})

Dataset({
    features: ['instruction', 'input', 'answer', 'messages', 'text'],
    num_rows: 29225
})
Dataset({
    features: ['instruction', 'input', 'answer', 'messages', 'text'],
    num_rows: 597
})


,prompt
0,"Объясни простыми словами, как работает Wi-Fi д..."
1,Напиши короткое и вежливое сообщение коллеге с...
2,"Подскажи, как распределить домашние дела, чтоб..."
3,"Объясни, как правильно сварить макароны, чтобы..."
4,Составь план подготовки к поездке в горы на вы...
5,"Подскажи, как выбрать недорогой подарок другу ..."
6,"Объясни, как хранить овощи и фрукты, чтобы они..."
7,"Напиши короткую инструкцию, что сделать перед ..."
8,"Подскажи, как ухаживать за комнатным растением..."
9,"Объясни, как быстро остудить горячий чай."


In [55]:
#baseline generation до обучения
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

def generate_answer(model, tokenizer, prompt, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

baseline_rows = []
for prompt in test_prompts:
    answer = generate_answer(base_model, tokenizer, prompt)
    baseline_rows.append({"prompt": prompt, "baseline_answer": answer})

baseline_df = pd.DataFrame(baseline_rows)
baseline_df

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

,prompt,baseline_answer
0,"Объясни простыми словами, как работает Wi-Fi д...",1. Время передачи данных: Wi-Fi работает на ос...
1,Напиши короткое и вежливое сообщение коллеге с...,Конечно! Вот примерный текст для переноса встр...
2,"Подскажи, как распределить домашние дела, чтоб...","1. Определите основные задачи: определите, что..."
3,"Объясни, как правильно сварить макароны, чтобы...",1. Выбери подходящий вид макарон: выбора завис...
4,Составь план подготовки к поездке в горы на вы...,1. Определите цель поездки: какую-то задачу ил...
5,"Подскажи, как выбрать недорогой подарок другу ...","1. Определите, что вы ищете в подарке.\n2. Поп..."
6,"Объясни, как хранить овощи и фрукты, чтобы они...",1. Выберите подходящий способ хранения: для ов...
7,"Напиши короткую инструкцию, что сделать перед ...","1. Проверьте, есть ли у вас все необходимые ве..."
8,"Подскажи, как ухаживать за комнатным растением...","1. Проверьте, что растение подходит для вашего..."
9,"Объясни, как быстро остудить горячий чай.",1. Выключите чайник.\n2. Включите холодильник....


In [56]:
baseline_df.loc[2]["baseline_answer"]

'1. Определите основные задачи: определите, что нужно сделать сегодня и завтра.\n\n2. Определите основные вещи: определите, что нужно сделать сегодня и завтра.\n\n3. Определите основные вещи: определите, что нужно сделать сегодня и завтра.\n\n4. Определите основные вещи: определите, что нужно сделать сегодня и завтра.\n\n5. Определите основные вещи: определите, что нужно сделать сегодня и завтра.\n\n6. Определите основные вещи: определите, что нужно сделать сегодня и завтра.\n\n7. Определите основные вещи: определите, что нужно сделать сегодня и завтра.\n\n8. Определите основные вещи: определите, что нужно сделать сегодня и завтра.\n\n9. Определите основные вещи: определите, что нужно сделать сегодня и завтра.\n\n10'

In [57]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [58]:
import json
import lm_eval

base_eval_results = lm_eval.simple_evaluate(
    model="hf",
    model_args=f"pretrained={MODEL_NAME},dtype=bfloat16",
    tasks=["arc_easy"],
    device="cuda:0",
    batch_size="auto",
    num_fewshot=0,
    apply_chat_template=True,
    log_samples=False,
)

print(json.dumps(base_eval_results["results"], indent=2, ensure_ascii=False))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Overwriting default num_fewshot of arc_easy from None to 0
Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running loglikelihood requests:   0%|          | 0/9501 [00:00<?, ?it/s]

Passed argument batch_size = auto:1. Detecting largest batch size


Running loglikelihood requests:   0%|          | 1/9501 [00:01<4:57:44,  1.88s/it]

Determined largest batch size: 64


Running loglikelihood requests: 100%|██████████| 9501/9501 [00:08<00:00, 1119.68it/s]


{
  "arc_easy": {
    "alias": "arc_easy",
    "acc,none": 0.5517676767676768,
    "acc_stderr,none": 0.010204645126856943,
    "acc_norm,none": 0.4238215488215488,
    "acc_norm_stderr,none": 0.01014000609521362
  }
}


In [59]:
#подготовка модели для qlora
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

gc.collect()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "up_proj",
        "down_proj",
        "gate_proj",
    ],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [60]:
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "qwen2_5_0_5b_ru_turbo_alpaca_qlora"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    report_to="none",
    seed=SEED,
    dataset_text_field="text",
    packing=False,
    completion_only_loss=False,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/29225 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/597 [00:00<?, ? examples/s]

In [61]:
train_result = trainer.train()
train_result

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
100,0.900811,0.889062
200,0.837873,0.859691
300,0.831804,0.841602
400,0.808626,0.828909
500,0.838349,0.817381
600,0.789669,0.811299
700,0.805794,0.807029
800,0.808564,0.805953
900,0.822131,0.805859
914,0.822131,0.805773


TrainOutput(global_step=914, training_loss=0.8504471956248878, metrics={'train_runtime': 962.2805, 'train_samples_per_second': 30.371, 'train_steps_per_second': 0.95, 'total_flos': 2.325762095513472e+16, 'train_loss': 0.8504471956248878})

In [62]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

('qwen2_5_0_5b_ru_turbo_alpaca_qlora/tokenizer_config.json',
 'qwen2_5_0_5b_ru_turbo_alpaca_qlora/chat_template.jinja',
 'qwen2_5_0_5b_ru_turbo_alpaca_qlora/tokenizer.json')

In [63]:
ft_rows = []
for prompt in test_prompts:
    answer = generate_answer(trainer.model, tokenizer, prompt)
    ft_rows.append({"prompt": prompt, "finetuned_answer": answer})

ft_df = pd.DataFrame(ft_rows)

compare_df = baseline_df.merge(ft_df, on="prompt", how="inner")
compare_df

,prompt,baseline_answer,finetuned_answer
0,"Объясни простыми словами, как работает Wi-Fi д...",1. Время передачи данных: Wi-Fi работает на ос...,"Wi-Fi - это технология, которая позволяет комп..."
1,Напиши короткое и вежливое сообщение коллеге с...,Конечно! Вот примерный текст для переноса встр...,"Привет, [имя коллеги], я хотел бы перенести на..."
2,"Подскажи, как распределить домашние дела, чтоб...","1. Определите основные задачи: определите, что...","Для того, чтобы убрать квартиру за один вечер,..."
3,"Объясни, как правильно сварить макароны, чтобы...",1. Выбери подходящий вид макарон: выбора завис...,"Для того, чтобы сварить макароны правильно, ну..."
4,Составь план подготовки к поездке в горы на вы...,1. Определите цель поездки: какую-то задачу ил...,"1. Проведите 3-4 часа прогулки по горам, чтобы..."
5,"Подскажи, как выбрать недорогой подарок другу ...","1. Определите, что вы ищете в подарке.\n2. Поп...",Если вы хотите выбрать недорогой подарок для д...
6,"Объясни, как хранить овощи и фрукты, чтобы они...",1. Выберите подходящий способ хранения: для ов...,"Чтобы овощи и фрукты дольше не портились, нужн..."
7,"Напиши короткую инструкцию, что сделать перед ...","1. Проверьте, есть ли у вас все необходимые ве...","Приходите из дома в дождливую погоду, чтобы не..."
8,"Подскажи, как ухаживать за комнатным растением...","1. Проверьте, что растение подходит для вашего...",Для начала нужно выбрать правильную почву и по...
9,"Объясни, как быстро остудить горячий чай.",1. Выключите чайник.\n2. Включите холодильник....,"Чтобы быстро остудить горячий чай, нужно снача..."


In [64]:
compare_df.to_csv("compare_df.csv", index=False)

In [65]:
ft_eval_results = lm_eval.simple_evaluate(
    model="hf",
    model_args=f"pretrained={MODEL_NAME},peft={OUTPUT_DIR},dtype=bfloat16",
    tasks=["arc_easy"],
    device="cuda:0",
    batch_size="auto",
    num_fewshot=0,
    apply_chat_template=True,
    log_samples=False,
)

print(json.dumps(ft_eval_results["results"], indent=2, ensure_ascii=False))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Overwriting default num_fewshot of arc_easy from None to 0
Chat template formatting change affects loglikelihood and multiple-choice tasks. See docs/chat-template-readme.md for details.
Running loglikelihood requests:   0%|          | 0/9501 [00:00<?, ?it/s]

Passed argument batch_size = auto:1. Detecting largest batch size
Determined largest batch size: 64


Running loglikelihood requests: 100%|██████████| 9501/9501 [00:15<00:00, 601.06it/s] 


{
  "arc_easy": {
    "alias": "arc_easy",
    "acc,none": 0.5833333333333334,
    "acc_stderr,none": 0.010116282977781398,
    "acc_norm,none": 0.5130471380471381,
    "acc_norm_stderr,none": 0.010256289925058391
  }
}


In [66]:
ft_metrics = {
    k: float(v)
    for k, v in ft_eval_results["results"]["arc_easy"].items()
    if k != "alias"
}

with open("lm_eval_ft_arc_easy.json", "w", encoding="utf-8") as f:
    json.dump(ft_metrics, f, ensure_ascii=False, indent=2)

metrics_compare = pd.DataFrame([
    {
        "metric": metric,
        "baseline": base_metrics.get(metric),
        "finetuned": ft_metrics.get(metric),
        "delta": ft_metrics.get(metric) - base_metrics.get(metric),
    }
    for metric in sorted(set(base_metrics) | set(ft_metrics))
])

metrics_compare

,metric,baseline,finetuned,delta
0,"acc,none",0.551768,0.583333,0.031566
1,"acc_norm,none",0.423822,0.513047,0.089226
2,"acc_norm_stderr,none",0.010140,0.010256,0.000116
3,"acc_stderr,none",0.010205,0.010116,-0.000088


In [68]:
#батч для профайлера
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader

profile_subset = train_dataset.select(range(64))

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=512,
        padding=False,
    )

profile_tokenized = profile_subset.map(
    tokenize_batch,
    batched=True,
    remove_columns=profile_subset.column_names,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

profile_loader = DataLoader(
    profile_tokenized,
    batch_size=8,
    shuffle=False,
    collate_fn=data_collator,
)

first_batch = next(iter(profile_loader))
{k: v.shape for k, v in first_batch.items()}

{'input_ids': torch.Size([8, 271]),
 'attention_mask': torch.Size([8, 271]),
 'labels': torch.Size([8, 271])}

In [69]:
#профайлинг нескольких шагов обучения
import torch
from torch.profiler import profile, ProfilerActivity

model.train()

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

profiler_table = None

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
    with_stack=False,
) as prof:
    for step, batch in enumerate(profile_loader):
        if step == 5:
            break

        batch = {k: v.to(model.device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)

        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        prof.step()

profiler_table = prof.key_averages().table(
    sort_by="cuda_time_total",
    row_limit=20,
)

print(profiler_table)

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
       autograd::engine::evaluate_function: MmBackward0         0.56%      17.078ms        40.35%        1.237s     734.028us       0.000us         0.00%     329.415ms     195.498us       1.88 KB           0 B      16.17 GB     -11.78 G

In [70]:
prof.export_chrome_trace("trace_qlora_training.json")

## Вывод

1) В работе была выбрана компактная instruction-tuned модель `Qwen/Qwen2.5-0.5B-Instruct` и русскоязычный instruction-датасет `IlyaGusev/ru_turbo_alpaca`.
2) Перед дообучением модель была проверена на ручной корзинке промптов и на задаче `arc_easy` из `lm-evaluation-harness`. По ручной оценке базовая модель уже умеет отвечать в нужном формате, но часто дает шаблонные, неестественные или некорректные ответы. На `arc_easy` базовые метрики составили `acc = 0.55` и `acc_norm = 0.42`.
3) Для дообучения использовался QLoRA-подход: базовая модель загружалась в 4-битном виде, а обучались только LoRA-адаптеры. Доля обучаемых параметров составила около `1.75%`, что хорошо соответствует идее parameter-efficient fine-tuning. В конфигурации LoRA были выбраны дефольные `r=16`, `lora_alpha=32`, `lora_dropout=0.05`, а адаптеры были добавлены в основные attention и MLP проекции. Такой набор параметров дал разумный компромисс между качеством, скоростью и устойчивостью обучения.
4) Во втором запуске итоговый train_loss составил `0.85`, а validation_loss снизился до `0.81`. Это указывает на то, что модель действительно адаптировалась к обучающему распределению, причём без признаков развала оптимизации. Валидационный loss на протяжении эпохи монотонно уменьшался и к концу обучения вышел на плато, что выглядит как нормальное завершение одной эпохи файнтюна.
5) После дообучения модель была повторно оценена на `arc_easy`. Метрики выросли до `acc = 0.58` и `acc_norm = 0.51`. Это можно считать улучшением качества на внешнем бенчмарке. При этом ручная оценка на корзинке показала более смешанную картину: часть ответов стала лучше по структуре и стилю, но на бытовых инструкциях улучшение оказалось неравномерным. Это важное наблюдение: уменьшение loss и рост benchmark-метрик не гарантируют одинаково сильный рост качества на произвольных пользовательских запросах.
6) Профилирование нескольких шагов обучения показало, что основное время уходит на матричные операции и их backward-проходы. Наиболее тяжёлыми операциями оказались `MmBackward0`, `aten::mm`, `aten::linear` и `aten::matmul`. Это означает, что главным узким местом во время fine-tuning являются линейные слои трансформера, а не токенизация или шаг оптимизатора. Отдельно виден вклад `MatMul4Bit` и `MatMul4BitBackward`, что ожидаемо для QLoRA: квантованные веса уменьшают память, но добавляют собственные вычислительные расходы.
7) Итоговый вывод по профилированию такой: обучение упирается прежде всего в матричные умножения, backward для линейных слоёв и накладные расходы квантованного 4-битного пайплайна. Это согласуется с архитектурой трансформеров и с тем, как работает QLoRA на практике. С точки зрения оптимизации скорости дальнейшие улучшения логично искать в уменьшении числа и длины последовательностей, подборе batch size, packing, а также в снижении лишних преобразований тензоров.